# **Notebook**: Download Building Inventory Data

__Date__: September, 2025  
__Author__: D. Acosta-Reyes  
__Supervisor__: J. Wartman  
__University of Washington__

__Content__: Code utilized to access and download building data.  
__Sources__:  
* Overture Footprint data: https://overturemaps.org/
* National Structures Inventory - NSI: https://www.hec.usace.army.mil/confluence/nsi

### Initial Setup

In [ ]:
pip install overturemaps

In [ ]:
pip install --upgrade overturemaps

In [ ]:
pip install lonboard

In [ ]:
'''Load and import necessary libraries'''
# General data libraries
import numpy as np
import pandas as pd
import geopandas as gpd

# Overture and mapping libraries
import overturemaps 
from shapely import wkb
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_categorical_cmap

# NSI and other download libraries
import re
from pathlib import Path
from urllib.parse import urljoin, urlparse
import requests

import os

# Define data path
data_path = "/Users/danielacosta/Library/CloudStorage/OneDrive-UW/0 - DA General Exam/Paper 1 - Exposure analysis/Data"

#### Use States boundaries for downloading

In [ ]:
# Download Census 2024 generalized state boundaries (GPKG ZIP)
census_url = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_500k.zip"
census_dir = os.path.join(data_path, "census_gov_data")
os.makedirs(census_dir, exist_ok=True)

census_zip_path = os.path.join(census_dir, os.path.basename(census_url))

if os.path.exists(census_zip_path):
    print(f"File already exists: {census_zip_path}")
else:
    urllib.request.urlretrieve(census_url, census_zip_path)
    print(f"Downloaded: {census_zip_path}")

# unzip data and load to use
states = gpd.read_file(f"zip://{census_zip_path}")

# Define CRS to use for Overture data (WGS84)
crs = "EPSG:4326"
states = states.to_crs(crs)

In [ ]:
# Exclude Alaska 'AK' and Hawaii 'HI' - CONUS only
states = states[~states['STUSPS'].isin(['AK', 'HI'])].reset_index(drop=True) 
# Separate Alaska and Hawaii for later use
ak = states[states['STUSPS'] == 'AK']
hi = states[states['STUSPS'] == 'HI']

# Add bbox tuple (minx, miny, maxx, maxy) for each state
states['bbox'] = (
    states.geometry
          .bounds[['minx', 'miny', 'maxx', 'maxy']]
          .round(6)
          .apply(lambda row: tuple(map(float, row.values)), axis=1)
)

ak['bbox'] = (
    ak.geometry
          .bounds[['minx', 'miny', 'maxx', 'maxy']]
          .round(6)
          .apply(lambda row: tuple(map(float, row.values)), axis=1)
)
hi['bbox'] = (
    hi.geometry
          .bounds[['minx', 'miny', 'maxx', 'maxy']]
          .round(6)
          .apply(lambda row: tuple(map(float, row.values)), axis=1)
)

#### Select data to download  
Comment (#) unused cells to define 'states' variable

In [ ]:
'''Download complete data for the US'''
# create bboxes list for Overture download -- All CONUS states 
bboxes = states['bbox'].tolist()
# check
print(bboxes)

In [ ]:
'''Download for specific states. First define the subset using NAME column. List of states by commas'''
states_subset = [
    'Alabama',
    'Washington'
]
# states = states[states['NAME'].isin(states_subset)].reset_index(drop=True)

# # create bboxes list for Overture download
# bboxes = states['bbox'].tolist()
# print(bboxes)

# 1) Overture Data

### Function to download Overture data

In [ ]:
# Function to fetch building records from Overture Building Data for given bounding boxes
def fetch_Overture_buildings(bboxes):
    """
    Fetch building records from Overture for selected bboxes.

    Args:
        bboxes (iterable): iterable of (minx, miny, maxx, maxy) tuples.

    Returns:
        GeoDataFrame in EPSG:4326 with fetched building records (may be empty).
    """
    
    gdfs = []
    
    # Use provided bboxes
    if bboxes is None:
        bboxes = states['bbox'].tolist()  # Use states' bboxes directly
    
    output_folder = os.path.join(data_path, "overture_buildings")
    os.makedirs(output_folder, exist_ok=True)
    print(f'Data will be saved to: {output_folder}')

    for i in range(len(bboxes)):
        try:
            bbox = bboxes[i]
            name = states['NAME'][i] if i < len(states['NAME']) else f"bbox_{i}"
            statefp = states['STATEFP'][i] if i < len(states['STATEFP']) else None
            print(f"Fetching bbox index {i+1}/{len(bboxes)} --({name})--: {bbox}")
        except Exception:
            print(f"Warning: invalid bbox index {i}; skipping")
            continue

        try:
            table = overturemaps.record_batch_reader("building", bbox).read_all()
            if table is None:
                continue
            table = table.combine_chunks()
            # keep only expected columns that exist
            cols = [c for c in ['id', 'height', 'source', 'version', 'subtype', 'class', 'geometry', 'wkb_geom'] if c in table.schema.names]
            table = table.select(cols)
            df = table.to_pandas()

            if df.empty:
                continue

            # Robust geometry handling: prefer 'wkb_geom', else handle 'geometry' column
            if 'wkb_geom' in df.columns:
                df['geometry'] = df['wkb_geom'].apply(lambda x: wkb.loads(x) if isinstance(x, (bytes, bytearray)) else None)
            elif 'geometry' in df.columns:
                # if geometry values are bytes (wkb), convert; otherwise assume already valid
                if df['geometry'].apply(lambda x: isinstance(x, (bytes, bytearray))).any():
                    df['geometry'] = df['geometry'].apply(lambda x: wkb.loads(x) if isinstance(x, (bytes, bytearray)) else None)
                # otherwise leave as-is (could be shapely already)
            # Drop rows where geometry conversion failed
            df = df.dropna(subset=['geometry']).reset_index(drop=True)
            # drop duplicate geometries
            df = df.drop_duplicates(subset=['geometry']).reset_index(drop=True)
            if df.empty:
                print("  --> No valid geometries after conversion.")
                continue
            gdf_local = gpd.GeoDataFrame(df.copy(), geometry='geometry', crs="EPSG:4326")

            # Attach state mask: keep only footprints that intersect the current state's geometry
            try:
                state_geom = states.at[i, 'geometry']
            except Exception:
                state_geom = None

            if state_geom is not None:
                # filter to features intersecting the state polygon
                # Use spatial index for efficient clipping
                gdf_local = gdf_local.clip(state_geom)
                gdf_local = gdf_local[gdf_local.geometry.notna()].copy()
                if gdf_local.empty:
                    continue

                # record the state association for each footprint
                gdf_local['STUSPS'] = states.at[i, 'STUSPS'] if 'STUSPS' in states.columns else name
                gdf_local['STATEFP'] = states.at[i, 'STATEFP'] if 'STATEFP' in states.columns else name

            print(f" --> Fetched {len(gdf_local)} building records for {name}")

            # save each iteration before appending to the list to avoid memory issues
            out_filename = f'{name}_overture_buildings.gpkg'
            output_path = os.path.join(output_folder, out_filename)
            gdf_local.to_file(output_path, driver='GPKG')
            # merge data
            gdfs.append(gdf_local)
            print(f" --> Records saved: {out_filename}")
        except Exception as e:
            print(f"Warning: failed to fetch bbox index {i} {bbox}: {e}")
            continue

    if gdfs:
        gdf_all = pd.concat(gdfs, ignore_index=True)
        gdf_all = gdf_all.set_geometry('geometry')
        gdf_all.crs = "EPSG:4326"
        print(f"Fetched a total of {len(gdf_all)} building records from Overture.")
        return gdf_all
    
    # return empty geodataframe with expected columns if nothing fetched
    return gpd.GeoDataFrame(columns=['id', 'height', 'source', 'version', 'subtype', 'class', 'geometry'], geometry='geometry', crs="EPSG:4326")

### Run Functions

#### a) CONUS Footprints

In [ ]:
# Fetching all buildings for CONUS from Overture
overture_gdf = fetch_Overture_buildings(bboxes)

# Sanity check:
print(overture_gdf.head(3))

In [ ]:
# save Complete Overture to file
out_filename = f'CONUS_overture_buildings.gpkg'
output_path = os.path.join(data_path, "overture_buildings", out_filename)
overture_gdf.to_file(output_path, driver='GPKG')

#### b) Alaska and Hawaii

In [ ]:
# Fetch Alaska and Hawaii separately using their specific bboxes
# Alaska
ak_bbox = ak['bbox'].iloc[0]
# modify third item of ak_bbox tupple to be -129.50000 to avoid fetching extraneous data from Russia and Canada, which are outside our study area and not relevant to our analysis
ak_bbox = (-179.146711, 51.215567, -129.500000, 71.387815)
# Hawaii
hi_bbox = hi['bbox'].iloc[0]

# Run Function
print(f"Fetching Alaska with bbox: {ak_bbox}")
ak_gdf = fetch_Overture_buildings(ak_bbox)
print(f"Fetching Hawaii with bbox: {hi_bbox}")
hi_gdf = fetch_Overture_buildings(hi_bbox)

In [ ]:
# save complete overture for AK and HI to file
ak_gdf.to_file(f'{data_path}/overture_buildings/AK_overture_buildings.gpkg', driver='GPKG')
hi_gdf.to_file(f'{data_path}/overture_buildings/HI_overture_buildings.gpkg', driver='GPKG')

# 2) NSI Data

In [ ]:
# Download NSI files listed at the NSI downloads page
nsi_url = "https://nsi.sec.usace.army.mil/downloads/nsi_2022/nsi_2022_"
out_dir = os.path.join(data_path, "nsi_buildings")
os.makedirs(out_dir, exist_ok=True)

# NSI download uses STATEFP state abbreviations in the file names -- CONUS only, so we exclude Alaska and Hawaii from the list of states to download
states_id = states[states['STATEFP']]['STATEFP'].tolist()

# Loop through the states and download the files
for i in states_id:
    # print name of state
    print(f'Downloading NSI data for: --{states[states["STATEFP"] == i]["NAME"].values[0]}-- (STATEFP: {i})')
    # URL for the download
    url = f"{nsi_url}{i}.gpkg.zip"
    # File path to save the downloaded file
    file_path = os.path.join(out_dir, f"nsi_2022_{i}.gpkg.zip")
    # Download the file
    try:       
        urllib.request.urlretrieve(url, file_path)
    except Exception as e:
        print(f"Failed to download {url}: {e}")
    # unzip data and load to merge complete dataset as gpkg (since geopandas can read gpkg directly from zip)
    nsi_gdf = gpd.read_file(file_path)
    nsi_gdf['STATEFP'] = i  # Add state identifier to the geodataframe
    print(f'Downloaded and loaded: {len(nsi_gdf)} records')
    if i == states_id[0]:
        nsi = nsi_gdf.copy()
    else:
        nsi = pd.concat([nsi, nsi_gdf], ignore_index=True)

print(f"Finished downloading NSI data for all states in the CONUS. Total records: {len(nsi)}")

In [ ]:
# Save complete NSI dataset to file
nsi.to_file(f'{out_dir}/US_nsi_2022.gpkg', driver='GPKG')

### End of Script